# Batch normalization and separable convolutions

Two patterns that cost almost nothing and change what is trainable — plus the ordering detail about BatchNormalization that most code gets wrong.

**Runs on:** CPU — about 4 minutes &nbsp;·&nbsp; **Slides:** [Chapter 9 — ConvNet Architecture Patterns](../../../course-web-slides/ch09/index.html) &nbsp;·&nbsp; **Section:** 03 — Batch normalization and 04 — Depthwise separable convolutions

---

## What batch normalization does

In [ ]:
import numpy as np
import keras
from keras import layers
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
activations = rng.normal(loc=3.0, scale=2.5, size=(1000, 64)).astype("float32")

bn = layers.BatchNormalization()
normed = bn(activations, training=True).numpy()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
a1.hist(activations.ravel(), bins=60); a1.set_title(
    f"before: mean {activations.mean():.2f}, std {activations.std():.2f}")
a2.hist(normed.ravel(), bins=60); a2.set_title(
    f"after: mean {normed.mean():.2f}, std {normed.std():.2f}")
plt.tight_layout(); plt.show()

Zero mean, unit variance, per feature, per batch. Chapter 6 said *normalize your inputs*; this normalizes the inputs to **every layer**, continuously, as the distribution shifts during training.

## The ordering that most code gets wrong

In [ ]:
# COMMON, and slightly wrong:
z = layers.Conv2D(32, 3, activation="relu")   # activation inside the conv
# ... then BatchNormalization after it

# BETTER:
inputs = keras.Input(shape=(32, 32, 3))
z = layers.Conv2D(32, 3, use_bias=False)(inputs)   # no activation, no bias
z = layers.BatchNormalization()(z)
z = layers.Activation("relu")(z)                   # activation AFTER
print("conv -> batchnorm -> activation:", z.shape)

Two details:

- **The activation goes after the normalization.** `relu` zeroes the negative half; normalizing afterwards is normalizing a truncated distribution. Doing it in this order gives `relu` a centred input, which is where it does the most work.
- **`use_bias=False`.** BatchNormalization has its own centring parameter, so the convolution's bias is redundant — parameters with nothing to do.

## Measuring whether it matters

In [ ]:
from keras.datasets import cifar10
(x, y), (xt, yt) = cifar10.load_data()
x = x.astype("float32") / 255
xt = xt.astype("float32") / 255

def build(use_bn, lr=1e-2):
    keras.utils.set_random_seed(0)
    i = keras.Input(shape=(32, 32, 3))
    z = i
    for f in [32, 64, 128]:
        z = layers.Conv2D(f, 3, padding="same", use_bias=not use_bn)(z)
        if use_bn:
            z = layers.BatchNormalization()(z)
        z = layers.Activation("relu")(z)
        z = layers.MaxPooling2D(2)(z)
    z = layers.GlobalAveragePooling2D()(z)
    o = layers.Dense(10, activation="softmax")(z)
    m = keras.Model(i, o)
    m.compile(optimizer=keras.optimizers.SGD(learning_rate=lr),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

plt.figure(figsize=(7, 4.2))
for use_bn in [False, True]:
    h = build(use_bn).fit(x[:20000], y[:20000], epochs=10, batch_size=128,
                          validation_split=.2, verbose=0)
    plt.plot(h.history["val_accuracy"], lw=1.6,
             label="with BatchNorm" if use_bn else "without")
    print(f"{'with' if use_bn else 'without':8s} BN -> "
          f"best val acc {max(h.history['val_accuracy']):.4f}")
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend()
plt.title("BatchNorm at a deliberately aggressive learning rate (SGD 1e-2)")
plt.show()

The gap widens with depth and with learning rate. **Batch normalization is what makes an aggressive learning rate survivable**, which is most of why it speeds training up.

## Freezing a model that contains BatchNormalization

> ⚠️ **The trap from chapter 8, in detail.** A frozen `BatchNormalization` layer still updates its running mean and variance during a forward pass in training mode — unless the layer itself is frozen. Setting `base.trainable = False` handles this correctly; freezing only the *weights* does not.

In [ ]:
base = keras.applications.VGG16(weights=None, include_top=False,
                                input_shape=(32, 32, 3))
base.trainable = False
print("trainable weights:", len(base.trainable_weights))
print("non-trainable:    ", len(base.non_trainable_weights))
print("\nBatchNormalization layers hold 2 trainable + 2 non-trainable weights:")
print("  gamma, beta   (learned)")
print("  moving_mean, moving_variance   (statistics, not gradients)")

## Separable convolutions

In [ ]:
def count(layer_fn, shape=(32, 32, 64)):
    i = keras.Input(shape=shape)
    o = layer_fn(i)
    return keras.Model(i, o).count_params()

regular = count(lambda z: layers.Conv2D(128, 3, padding="same")(z))
separable = count(lambda z: layers.SeparableConv2D(128, 3, padding="same")(z))

print(f"Conv2D(128, 3)          {regular:>8,} parameters")
print(f"SeparableConv2D(128, 3) {separable:>8,} parameters")
print(f"{regular/separable:.1f}x fewer")

Expected output:

```
Conv2D(128, 3)            73,856 parameters
SeparableConv2D(128, 3)    8,896 parameters
8.3x fewer
```

A regular convolution learns spatial **and** channel patterns together. A separable one splits them: a depthwise convolution over each channel independently, then a 1×1 convolution to mix channels.

The assumption — that spatial and channel structure are largely independent — is a **stronger prior**, and on natural images it holds. Chapter 12's YOLO and chapter 17's U-Net both rely on it.

## Does the assumption cost accuracy?

In [ ]:
def build_sep(separable):
    keras.utils.set_random_seed(0)
    Conv = layers.SeparableConv2D if separable else layers.Conv2D
    i = keras.Input(shape=(32, 32, 3))
    z = layers.Conv2D(32, 3, padding="same", activation="relu")(i)  # first stays regular
    for f in [64, 128]:
        z = Conv(f, 3, padding="same", use_bias=False)(z)
        z = layers.BatchNormalization()(z)
        z = layers.Activation("relu")(z)
        z = layers.MaxPooling2D(2)(z)
    z = layers.GlobalAveragePooling2D()(z)
    o = layers.Dense(10, activation="softmax")(z)
    m = keras.Model(i, o)
    m.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

for sep in [False, True]:
    m = build_sep(sep)
    h = m.fit(x[:20000], y[:20000], epochs=10, batch_size=128,
              validation_split=.2, verbose=0)
    print(f"{'separable' if sep else 'regular  '}: "
          f"{m.count_params():>8,} params, "
          f"best val acc {max(h.history['val_accuracy']):.4f}")

Comparable accuracy from a fraction of the parameters. Note that the **first** layer stays a regular convolution — with three input channels there is almost nothing to separate, and the assumption has no purchase.

---

## What to take away

- Batch normalization keeps each layer's inputs centred as the distribution shifts during training.
- **Conv (no bias) → BatchNorm → Activation** is the right order, and most code gets it wrong.
- Freeze with `layer.trainable = False`, or the running statistics keep updating.
- Separable convolutions assume spatial and channel structure are independent — eight times fewer parameters, comparable accuracy.